### 炭素構造のクラスタリング

ここではクラスタリングしてから、
可視化のために次元圧縮をしている。

**データ取得からデータ解析**

In [ ]:
from typing import List, Tuple
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)


In [ ]:
g_action = ["km", "gmm"]# kmeanとGMMを行う。
g_nclusters = 3

In [ ]:
def select_structure(df):
    """select structures

    Args:
        df (pd.DataFrame): data

    Returns:
        pd.DataFrame: selected data
    """
    keylist = []
    for key in df.index:
        s = key.split("-")
        dim = s[0]
        i = int(s[1])
        if (dim == "3D" or dim == "2D") and i <= 5:
            keylist.append(key)
    return df.loc[keylist]


In [ ]:
# データ取得
def add_sptype(df):
    """dataframeにnnatom_strを加える。

    Args:
        df (pd.DataFrame): データ。

    Returns:
        pd.DataFrame: nnatom_strを加えたデータ
    """
    label_str = {2.0:"sp", 2.5: "sp2_edge", 3.0: "sp2", 
                 3.5: "sp2_tube", 4.0:"sp3"} 
    df["nnatom_str"] = [label_str[x] for x in df["nnatom"]]
    return df

def get_data():
    """carbon8のデータを得る。

    Returns:
        pd.DataFrame: nnatom_strがあるデータ。
        pd.DataFrame: 全データ。
        [str]: 説明変数名リスト。
    """
    df_select = pd.read_csv(
        "../data_calculated/Carbon8_descriptor_selected.csv", index_col=[0, 1])
    df_select = add_sptype(df_select)
    
    descriptor_names = ['a0.25_rp1.0', 'a0.25_rp1.5', 'a0.25_rp2.0', 'a0.25_rp2.5',
                         'a0.25_rp3.0', 'a0.5_rp1.0', 'a0.5_rp1.5', 'a0.5_rp2.0', 'a0.5_rp2.5',
                         'a0.5_rp3.0', 'a1.0_rp1.0', 'a1.0_rp1.5', 'a1.0_rp2.0', 'a1.0_rp2.5',
                         'a1.0_rp3.0']
    # Xraw_select = df_select.loc[:,descriptor_names].values
    nnatom = df_select.loc[:,"nnatom"]
    nnlabel_str = {2.0:"sp", 2.5: "sp2_edge", 3.0: "sp2", 
             3.5: "sp2_tube", 4.0:"sp3"} 

    df_all = pd.read_csv("../data_calculated/Carbon8_descriptor.csv", index_col=[0,1])
    # Xraw = df.loc[:,descriptor_names].values

    return df_select, df_all, descriptor_names

g_df_select, g_df_all, g_descriptor_names = get_data()

In [ ]:
g_df_select

In [ ]:
def add_scaledX(df : pd.DataFrame, descriptor_names: List[str], scaler :StandardScaler=None) ->Tuple[pd.DataFrame, List[str]]:
    """説明変数の規格化を行う。

    scalerがNoneであれば、scalerを作成する。

    Args:
        df (pd.DataFrame): データ
        descriptor_names (List[str]): 説明変数カラム名
        scaler (StandardScaler, optional): StandardScaler instance. Defaults to None.

    Returns:
        pd.DataFrame: データ。
        List[str]: 規格化した説明変数名リスト。
        StandardScaler: StandardScaler instance.
    """
    df = df.copy()
    
    # make labels of normalized values
    X_labels = []
    for i in descriptor_names:
        X_labels.append("s_{}".format(i))
    
    if X_labels[0] in df.columns:
        return df, X_labels, scaler
    
    #データ加工
    Xraw = df[descriptor_names].values
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(Xraw)
    X = scaler.transform(Xraw)    
    
    df_scaledX = pd.DataFrame(X,index=df.index, columns=X_labels)
    
    return pd.concat([df,df_scaledX], axis=1), X_labels, scaler

g_df_all, g_X_labels, g_scaler = add_scaledX(g_df_all, g_descriptor_names)
g_df_select, _, _ = add_scaledX(g_df_select, g_descriptor_names, g_scaler)

クラスタリングは以下のように行う。

In [ ]:
def apply_clustering(df, X_labels,  action, nclusters=3,km=None, gmm=None):
    """クラスタリングを行う。

    Args:
        df (pd.DataFrame): データ。
        X_labels ([str]]): 説明変数カラム名リスト
        action ([str]]): km and/or gmm
        nclusters (int, optional): クラスタ数. Defaults to 3.
        km (KMeans, optional): KMenas instance. Defaults to None.
        gmm (GaussianMixture, optional): GaussianMixtureインスタンス. Defaults to None.

    Returns:
        [type]: [description]
    """
    df = df.copy()
    X = df[X_labels].values
    km = None
    gmm = None
    # データ解析, 3 class
    if "km" in action:
        if km is None:
            km = KMeans(nclusters)
            km.fit(X)
        yp_km = km.predict(X)
        df["km"] = yp_km
    if "gmm" in action:
        if gmm is None:
            gmm = GaussianMixture(nclusters)
            gmm.fit(X)
        yp_gmm = gmm.predict(X)
        yproba_gmm = gmm.predict_proba(X)
        df["gmm"] = yp_gmm
        df["yproba_gmm"] = yproba_gmm.tolist()
    return df, km, gmm

g_df_all, g_km, g_gmm = apply_clustering(g_df_all, g_X_labels, g_action, nclusters=g_nclusters)
g_df_select, _, _  = apply_clustering(g_df_select, g_X_labels, g_action, nclusters=g_nclusters, km=g_km,gmm=g_gmm)


In [ ]:
g_df_all


分けやすい次元で分ける。

次元圧縮の可視化の軸で分ける。


**可視化**

可視化を含めて解析していく。
二次元で可視化するためにPCAにより次元圧縮を行っている。
これも次元圧縮の用途の一つである。

In [ ]:
def add_pca(df, X_labels, ndim=2, drd=None, pca_prefix="pca"):
    """pca{1..ndim}を加える

    Args:
        df (pd.DataFrame): データ。
        X_labels ([str]]): 説明変数カラム名リスト。 
        ndim (int, optional): 次元削減後の次元数. Defaults to 2.
        drd (PCA, optional): PCA instance. Defaults to None.
        pca_prefix (str, optional): PCAをおこなった説明変数のカラム名のprefix。 Defaults to "pca".

    Returns:
        pd.DataFrame: データ。
        [str]: pcaを行ったカラム名のリスト。
        PCA: PCA instance.
    """
    df = df.copy()
    pca_labels = []
    for i in range(ndim):
        pca_labels.append("{}{}".format(pca_prefix, i+1))
    
    if pca_labels[0] in df.columns:
        return df, pca_labels, drd
    
    X = df[X_labels].values
    if drd is None:    
        drd = PCA(ndim)
        drd.fit(X)
    X2 = drd.transform(X)
    df_pca = pd.DataFrame(X2, index=df.index, columns=pca_labels)
    
    return pd.concat([df,df_pca], axis=1), pca_labels, drd

# 可視化のために二次元に変換する。
g_df_all, g_pca_labels, g_pca = add_pca(g_df_all, g_X_labels)
g_df_select, _, _ = add_pca(g_df_select, g_X_labels, drd=g_pca)

In [ ]:
g_df_select

In [ ]:
def plot_X2(df_all, df_select, pca_labels, target_name, filename=None):
    """plot descriptor with yp classes

    Args:
        X2 (np.data): descriptor
        yp (np.array): target classes
        filename (str, optional): filename to save data. Defaults to None.
    """
    X2 = df_all[pca_labels].values
    yp = df_all[target_name].values
    
    nnatom = df_select["nnatom_str"].values
    X2_select = df_select[pca_labels].values
    yp_select = df_select[target_name].values
    
    marker = ["+", "o", "x"]
    # color = ["red", "blue", "green"]
    color = plt.cm.get_cmap("Pastel1")
    color_select = plt.cm.get_cmap("Set1")
    xlim = X2[:, 0].min()-0.1, X2[:, 0].max()+0.1
    ylim = X2[:, 1].min()-0.1, X2[:, 1].max()+0.1
    plt.figure()
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("PCA1")
    plt.ylabel("PCA2")

    for y in np.unique(yp):
        index = yp == y
        Xtmp = X2[index, :]
        plt.scatter(Xtmp[:, 0], Xtmp[:, 1],  cmap=color, s=1)

    for i,nn in enumerate(np.unique(nnatom)):
        index = nnatom == nn
        Xtmp = X2_select[index, :]
        plt.scatter(Xtmp[:, 0], Xtmp[:, 1], cmap=color_select,label=nn)        
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0,)

    if filename is not None:
        plt.savefig(filename)
    plt.show()

plot_X2(g_df_all, g_df_select, g_pca_labels, "km") 
#    X2, yp_km, X2_select, yp_km_select, nnatom)

In [ ]:
plot_X2(g_df_all, g_df_select, g_pca_labels, "gmm", 
        filename="image_executed/carbon8_gmm_clusters.png")

gaussian mixture modelを用いると予測値が確率でも出力される。
次は、●の濃淡でその確率を示す。

In [ ]:
import seaborn as sns


def plot_contour(df, columns, nclusters, yproba_label):
    """scatter plot with probabilities

    Args:
        X2 (np.array): data
        yproba (np.array): probabilities
    """
    X2 = df[columns].values
    yproba = np.array(df[yproba_label].values.tolist())
    
    plt.figure()
    for i in range(nclusters):
        colors = ["red", "blue", "green", "purple","yellow"]
        fig, ax = plt.subplots()
        cmap = sns.light_palette(colors[i], as_cmap=True)
        plot = plt.scatter(X2[:, 0], X2[:, 1], marker=".",
                           c=yproba[:, i], cmap=cmap, label=str(i))
        fig.legend()
        fig.colorbar(plot)
        fig.show()


plot_contour(g_df_all, g_pca_labels, nclusters=g_nclusters, yproba_label="yproba_gmm")


### 問題１

次元圧縮してからクラスタリングを行う。


### 付録１

BICによる成分数の選択方法

https://scikit-learn.org/stable/auto_examples/mixture/plot_gmm_selection.html


### 付録２

elbow法

Fe2クラスターで行った．
